## Rename demultiplexed FGCZ samples 

In [1]:
import pandas as pd
import glob
from pathlib import Path

fasta_files = glob.glob('../data/fastqs/*.fastq.gz')
fasta_files = [Path(f).name for f in fasta_files]
fasta_files[:5]

['o35218.bc1005--bc1033.fastq.gz',
 'o35218.bc1005--bc1035.fastq.gz',
 'o35218.bc1005--bc1044.fastq.gz',
 'o35218.bc1005--bc1045.fastq.gz',
 'o35218.bc1005--bc1076.fastq.gz']

In [2]:
barcode_mapping = pd.read_excel('sample_sheet.xlsx', index_col=None)
barcode_mapping[['barcode_fwd', 'barcode_rev']] = barcode_mapping['barcode'].str.split('--', expand=True)
barcode_mapping.head()

,barcode,Sample name,fastq name,timepoint,barcode_fwd,barcode_rev
0,bc1005--bc1033,A8-03_TnpB_L0,o35218.bc1005--bc1033.fastq.gz,0,bc1005,bc1033
1,bc1007--bc1033,A8-03_TnpB_L1,o35218.bc1007--bc1033.fastq.gz,1,bc1007,bc1033
2,bc1008--bc1033,A8-03_TnpB_L2,o35218.bc1008--bc1033.fastq.gz,2,bc1008,bc1033
3,bc1012--bc1033,A8-03_TnpB_L3,o35218.bc1012--bc1033.fastq.gz,3,bc1012,bc1033
4,bc1015--bc1033,A8-03_TnpB_L4,o35218.bc1015--bc1033.fastq.gz,4,bc1015,bc1033


In [10]:

import re

def find_sample_name(fasta_files, barcode_mapping):
    pattern = r"o\d+\.(bc\d+)--(bc\d+)\.fastq.gz"

    mapping_data = []

    for filename in fasta_files:
        match = re.search(pattern, filename)
        if match:
            forward_barcode_id = match.group(1)
            reverse_barcode_id = match.group(2)

            sample_row = barcode_mapping[
                (barcode_mapping['barcode_fwd'] == forward_barcode_id) &
                (barcode_mapping['barcode_rev'] == reverse_barcode_id)
            ]
            if sample_row.empty:
                print(
                    f"Sample name not found for {filename}. "
                    f"Forward Barcode ID: {forward_barcode_id}, "
                    f"Reverse Barcode ID: {reverse_barcode_id}"
                )
            else:
                sample_name = sample_row['Sample name'].iloc[0]
                sample_fastq = sample_name + '.fastq.gz'
                mapping_data.append({"fgcz_fastq": filename, "renamed_fastq": sample_fastq, 'sample': sample_name})

    return pd.DataFrame(mapping_data)

sample_info = find_sample_name(fasta_files, barcode_mapping)
sample_info

Sample name not found for o35218.bc1005--bc1076.fastq.gz. Forward Barcode ID: bc1005, Reverse Barcode ID: bc1076
Sample name not found for o35218.bc1005--bc1082.fastq.gz. Forward Barcode ID: bc1005, Reverse Barcode ID: bc1082
Sample name not found for o35218.bc1005--bc1083.fastq.gz. Forward Barcode ID: bc1005, Reverse Barcode ID: bc1083
Sample name not found for o35218.bc1005--bc1089.fastq.gz. Forward Barcode ID: bc1005, Reverse Barcode ID: bc1089
Sample name not found for o35218.bc1005--bc1096.fastq.gz. Forward Barcode ID: bc1005, Reverse Barcode ID: bc1096
Sample name not found for o35218.bc1005--bc1098.fastq.gz. Forward Barcode ID: bc1005, Reverse Barcode ID: bc1098
Sample name not found for o35218.bc1005--bc1100.fastq.gz. Forward Barcode ID: bc1005, Reverse Barcode ID: bc1100
Sample name not found for o35218.bc1005--bc1101.fastq.gz. Forward Barcode ID: bc1005, Reverse Barcode ID: bc1101
Sample name not found for o35218.bc1007--bc1076.fastq.gz. Forward Barcode ID: bc1007, Reverse Ba

,fgcz_fastq,renamed_fastq,sample
0,o35218.bc1005--bc1033.fastq.gz,A8-03_TnpB_L0.fastq.gz,A8-03_TnpB_L0
1,o35218.bc1005--bc1035.fastq.gz,A8-03_TnpB_L8.fastq.gz,A8-03_TnpB_L8
2,o35218.bc1005--bc1044.fastq.gz,A8-03_TnpB_L16.fastq.gz,A8-03_TnpB_L16
3,o35218.bc1005--bc1045.fastq.gz,A8-03_TnpB_L24.fastq.gz,A8-03_TnpB_L24
4,o35218.bc1007--bc1033.fastq.gz,A8-03_TnpB_L1.fastq.gz,A8-03_TnpB_L1
5,o35218.bc1007--bc1044.fastq.gz,A8-03_TnpB_L17.fastq.gz,A8-03_TnpB_L17
6,o35218.bc1007--bc1045.fastq.gz,A8-03_TnpB_L25.fastq.gz,A8-03_TnpB_L25
7,o35218.bc1008--bc1033.fastq.gz,A8-03_TnpB_L2.fastq.gz,A8-03_TnpB_L2
8,o35218.bc1008--bc1035.fastq.gz,A8-03_TnpB_L10.fastq.gz,A8-03_TnpB_L10
9,o35218.bc1008--bc1044.fastq.gz,A8-03_TnpB_L18.fastq.gz,A8-03_TnpB_L18


In [11]:
sample_info = sample_info.merge(barcode_mapping[['Sample name', 'timepoint']], left_on='sample', right_on='Sample name').drop(columns='Sample name')
sample_info.to_csv('fgcz_sample_map.csv')
sample_info.set_index('sample').to_excel('samples_info.xlsx')
sample_info

,fgcz_fastq,renamed_fastq,sample,timepoint
0,o35218.bc1005--bc1033.fastq.gz,A8-03_TnpB_L0.fastq.gz,A8-03_TnpB_L0,0
1,o35218.bc1005--bc1035.fastq.gz,A8-03_TnpB_L8.fastq.gz,A8-03_TnpB_L8,8
2,o35218.bc1005--bc1044.fastq.gz,A8-03_TnpB_L16.fastq.gz,A8-03_TnpB_L16,16
3,o35218.bc1005--bc1045.fastq.gz,A8-03_TnpB_L24.fastq.gz,A8-03_TnpB_L24,24
4,o35218.bc1007--bc1033.fastq.gz,A8-03_TnpB_L1.fastq.gz,A8-03_TnpB_L1,1
5,o35218.bc1007--bc1044.fastq.gz,A8-03_TnpB_L17.fastq.gz,A8-03_TnpB_L17,17
6,o35218.bc1007--bc1045.fastq.gz,A8-03_TnpB_L25.fastq.gz,A8-03_TnpB_L25,25
7,o35218.bc1008--bc1033.fastq.gz,A8-03_TnpB_L2.fastq.gz,A8-03_TnpB_L2,2
8,o35218.bc1008--bc1035.fastq.gz,A8-03_TnpB_L10.fastq.gz,A8-03_TnpB_L10,10
9,o35218.bc1008--bc1044.fastq.gz,A8-03_TnpB_L18.fastq.gz,A8-03_TnpB_L18,18


In [11]:
def create_symlinks(samples, src_dir, dest_dir):
    for idx, row in samples.iterrows():
        src_file_path = (src_dir / row['fgcz_fastq']).resolve()
        dest_file_path = dest_dir / f"{row['renamed_fastq']}"

        # Check if the source file exists
        if src_file_path.exists():
            # Create a symlink if the destination link doesn't exist yet
            if not dest_file_path.exists():
                dest_file_path.symlink_to(src_file_path.resolve())
            else:
                print(f"Symlink already exists for {dest_file_path}")
        else:
            print(f"Source file does not exist: {src_file_path}")

create_symlinks(sample_info, src_dir=Path('../data/fastqs'), dest_dir=Path('../data/fastqs_renamed'))